In [36]:
import transformers 
from huggingface_hub import HfApi
import os 

cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
api = HfApi()

print("\nLocal Hugging Face Cached Models and Their Tasks:\n")

for item in os.listdir(cache_dir):
    if item.startswith("models--"):
        # Convert cache folder name to repo_id
        repo_id = item.replace("models--", "").replace("--", "/")

        try:
            info = api.model_info(repo_id)
            tasks = info.pipeline_tag if info.pipeline_tag else "unknown"
        except Exception as e:
            tasks = "unknown"

        print(f"{repo_id}  -->  {tasks}")


Local Hugging Face Cached Models and Their Tasks:

albert-base-v2  -->  fill-mask
coqui/XTTS-v2  -->  text-to-speech
dandelin/vilt-b32-finetuned-vqa  -->  visual-question-answering
distilbert-base-uncased-finetuned-sst-2-english  -->  text-classification
distilgpt2  -->  text-generation
facebook/detr-resnet-50-panoptic  -->  image-segmentation
facebook/mms-tts-eng  -->  text-to-speech
google/flan-t5-small  -->  unknown
google/vit-base-patch16-224  -->  image-classification
guillaumekln/faster-whisper-small  -->  automatic-speech-recognition
Helsinki-NLP/opus-mt-en-hi  -->  translation
Helsinki-NLP/opus-mt-en-mr  -->  translation
Helsinki-NLP/opus-mt-hi-en  -->  translation
Helsinki-NLP/opus-mt-mr-en  -->  translation
HuggingFaceH4/zephyr-7b-alpha  -->  text-generation
hustvl/yolos-tiny  -->  object-detection
microsoft/speecht5_hifigan  -->  unknown
microsoft/speecht5_tts  -->  text-to-speech
MIT/ast-finetuned-audioset-10-10-0.4593  -->  audio-classification
nlpconnect/vit-gpt2-image-c

In [ ]:
Runnable Types :
    1] RunnableMap
    2] RunnableSequence
    3] RunnableLamda
    4] RunnableParallel
    5] RunnableBranch
    6]RunnablePassthrough
    


In [ ]:
langchain expression  language (LCEL) most important  in langchain.
- it let you build pipelines in a clean  chainbale way

prompt -> model -> output parser

before LCEL chain where messy

but now -> output = ( prompt | model | parser ).invoke(input)


In [ ]:
Why Runnable ?

- prompt template, llm , output parser , function, custom python function, any combination (chain)


Runnable Components:
     - RunnableSequence : Chain steps together
	- RunnableParallel : Run steps side by side
	- RunnableLambda : Add custom py function
	- RunnableMap : apply  mapping or branching
	- invoke() : single call
	- stream() :  stream  tokens
	- batch() : handle multiple inputs

In [ ]:
# minimal  LCEL pipeline RunnableSequence used

from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from transformers import  AutoTokenizer, AutoModelForCausalLM, pipeline

#model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu")

#pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

llm = HuggingFacePipeline(pipeline=pipe)

#prompt
prompt = PromptTemplate.from_template("Explain {topic} in simple words")
parser = StrOutputParser()

#chain
chain = prompt | llm | parser

result = chain.invoke({"topic":"gravity"})
print(result)

In [ ]:
from langchain.schema.runnable import RunnableMap

chain = RunnableMap({
	"short": prompt | llm,
	"detialed": prompt | llm
})

result = chain.invoke({"topic": "black holes"})
print(result)

In [ ]:
# minimal  LCEL pipeline Basic chain used

from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from transformers import  AutoTokenizer, AutoModelForCausalLM, pipeline

#model - llm
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu")

#pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

llm = HuggingFacePipeline(pipeline=pipe)

#prompt
prompt = PromptTemplate.from_template("Explain {topic} in simple words")
parser = StrOutputParser() #parser

#chain
chain = prompt | llm | parser

result = chain.invoke({"topic":"gravity"})
print(result)

Device set to use cpu
C:\Users\Sumit\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


Explain gravity in simple words which allows the human body greater force than other organs.” These results may suggest that the ability to overcome gravity is something that is not previously thought to have exist in our body.




But the


In [ ]:
#RunableMap

- run multiple operation parallel and returns directory of results
- input -> {task1, task2, task3}

1] good for  summary and keywords at same time
2] creating multiple variation of output
3] preprocessing  + LLM response
4] combing  embedding  + generation

In [ ]:
#RunnableMap
from  langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_core.output_parsers import StrOutputParser

from langchain.schema.runnable import RunnableMap

model_name = "distilgpt2"

model = AutoModelForCausalLM.from_pretrained(model_name,device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

pipe = pipeline("text-generation", model=model,tokenizer=tokenizer)

llm = HuggingFacePipeline(pipeline=pipe)

prompt = PromptTemplate(
    input_variable=["Topic"],template="Explain {topic} in simple words"
)

parser = StrOutputParser()


chain = RunnableMap({"short": prompt|llm, "detailed": prompt|llm})

result = chain.invoke({"topic":"black hole"})
result

Device set to use cpu


{'short': "Explain black hole in simple words, this is a beautiful thing to think about. You've discovered that a black hole isn't that big or bigger than earth. It doesn't have a hole in the middle from the center, and it's in",
 'detailed': 'Explain black hole in simple words, is an exotic material. You create a special object that you claim to be an object. You place it on top of a hole, enclosing it with a special image. A hole you attach to it is'}

In [ ]:
# example 2: runnable map

from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_core.runnables import  RunnableMap, RunnableLambda

model_name = "distilgpt2"

model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe= pipeline('text-generation', model=model, tokenizer=tokenizer)

llm = HuggingFacePipeline(pipeline=pipe)

def summarize_text(x):
    return x[:60] + "..."

def extract_keywords(x):
    words = x.split()
    return list(set(words[:5]))


runnable = RunnableMap(
            summary= RunnableLambda(summarize_text),
            keywords = RunnableLambda(extract_keywords),
            llm_output = llm
)

input = "Realtivty  explain how space and time  are connected"
runnable.invoke(input)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'summary': 'Realtivty  explain how space and time  are connected...',
 'keywords': ['explain', 'and', 'how', 'Realtivty', 'space'],
 'llm_output': 'Realtivty  explain how space and time  are connected to the universe.\n\nThe universe is made up of a sphere formed on a circular plane, which is defined on the bottom of the sphere. This is the space of an object'}

In [15]:
# RunnableSequnce

from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline, AutoTokenizer,AutoModelForCausalLM

from langchain.schema.runnable import RunnableSequence


model_name = "distilgpt2"

model = AutoModelForCausalLM.from_pretrained(model_name,device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

llm = HuggingFacePipeline(pipeline=pipe)

prompt = PromptTemplate(input_varible=["topic"] , template= "explain {topic} complex way")

parser = StrOutputParser()

chain = RunnableSequence(first=prompt, middle=[llm], last=parser)

chain.invoke({"topic": "Hello world program in python"})

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'explain Hello world program in python complex way, and python.py. A little easier read: python.py.\n\nConclusion A Simple, Simple Python Programming Language\nPython is my main language, and it has a huge advantages for programming language'

In [ ]:
3] RunnableLamda and lambda

In [ ]:
#lambda argument : expression
#lightweight , does not run asynchronously, does not integrate  with langhchain chains

square = lambda X: X*X
print(square(5))

25


In [ ]:
RunnableeLamdba
- converts python  function  into components that fits into LCEL chains

why runnbaleLambda  ??
- basic langchain pipeline -> prompt | llms | parser

IF you  want custom  python logic  in  middle  you need  runnablelamdba not normal  lambda

1] preporcessing  input
2] postporcessing llms output
3] clean text
4] convert output to json
5] split text
6] exact numbers

In [ ]:
# RunnableLambda preprocessing input

from langchain_core.runnables import RunnableLambda

def preprocess(text):
    return text.lower().strip()


clean =  RunnableLambda(preprocess)

print(clean.invoke("   HELLO WORLD   "))

hello world


In [10]:
#RunnableMap : postprocess llm output

from langchain.llms import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda
from transformers import pipeline , AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

model = AutoModelForCausalLM.from_pretrained(model_name,device_map="cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=pipe)

def postprocess(text):
    return text.replace("\n"," ").strip()

runnable = RunnableLambda(postprocess)

text = llm("explain about me")
clean = runnable.invoke(text)

print("raw: ",text)
print("clean: ",clean)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


raw:  explain about me. It’s the same.’’’’

What you don’t know, is that I was there to kill a guy and kill someone in the back while you were asleep.
clean:  explain about me. It’s the same.’’’’  What you don’t know, is that I was there to kill a guy and kill someone in the back while you were asleep.


NameError: name 'langchain' is not defined

In [2]:
#runnablelamdba : clean text 
import re 
from langchain_core.runnables import  RunnableLambda

def clean_text(x):
    return re.sub("[^a-zA-Z0-9 ]","",x)

clean = RunnableLambda(clean_text)
print(clean.invoke("HEllo @@ Relativity"))

HEllo  Relativity


In [3]:
#RunnableLambda : Convert ouptut to json

import json
from langchain_core.runnables import RunnableLambda

def to_json(x):
    return json.dumps({"result":x}, indent=2)

json_convert  = RunnableLambda(to_json)
print(json_convert.invoke("Relativity connects sapce and time"))

{
  "result": "Relativity connects sapce and time"
}


In [4]:
#RunnableLambda : Split text

from langchain_core.runnables import RunnableLambda

def split_sentence(x):
    return x.split(".")

splitter = RunnableLambda(split_sentence)
print(splitter.invoke("Relavity is impoortant. It changes physics"))

['Relavity is impoortant', ' It changes physics']


In [6]:
# RunnableLambda : Extra Numbers 

import re
from langchain_core.runnables import RunnableLambda

def extra_num(x):
    return re.findall(r"\d+",x)

extractor = RunnableLambda(extra_num)
print(extractor.invoke("Einstein published realitivity in 1905 and 1915"))

['1905', '1915']


In [23]:
"""Runnablelamdba : preprocessing , llm output , post porcessing """

import re
from langchain.llms import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

#load model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation",model=model,tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=pipe)

#preprocess
pre = RunnableLambda(lambda x: x.lower().strip())

#postproces
post = RunnableLambda(lambda x: x.replace("\n"," ").strip())

#clean 
def clean_text(x):
    return re.sub("[^a-zA-Z0-9 ]","",x)

clean = RunnableLambda(clean_text)

#Run
input = pre.invoke("   EXPLAIN RELATIVITY   ")
raw_output = llm(input)
post_output = post.invoke(raw_output)
clean_output = clean.invoke(post_output)

print("raw: ",raw_output)
print("post: ",post_output)
print("clean:", clean_output)

Device set to use cpu


raw:  explain relativity: the first quantum experiment
What other topics have we asked in our work?
1) Quantum experiments, or experiments, in quantum field theories, or experiments, or experiments
2) The scientific fields are important. Science has several
post:  explain relativity: the first quantum experiment What other topics have we asked in our work? 1) Quantum experiments, or experiments, in quantum field theories, or experiments, or experiments 2) The scientific fields are important. Science has several
clean: explain relativity the first quantum experiment What other topics have we asked in our work 1 Quantum experiments or experiments in quantum field theories or experiments or experiments 2 The scientific fields are important Science has several


In [ ]:
2] RunnableMap

In [7]:
#generate summary + keyword together
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu")
pipe = pipeline("text-generation",model=model,tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=pipe)

summary = RunnableLambda(lambda text: llm(f"Sumarize: {text}"))
keywords = RunnableLambda(lambda text: llm(f"Extract keywords: {text}"))


#RunnableMap 
combined = RunnableMap({
    "summary": summary, "keywords":keywords
})

output = combined.invoke("Artifical Intelligence ")
print(output)

Device set to use cpu
C:\Users\Sumit\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
C:\Users\Sumit\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'summary': 'Sumarize: Artifical Intelligence 【+] +1【+] +1【+] +3【+] +2【+] +3【+] +2【+] +3【+] +3【', 'keywords': 'Extract keywords: Artifical Intelligence ________\n\n\n\n\nThese are your thoughts and suggestions on each section.\n\nTo start off with, click in "Sessions" on the left.\nClick on "Tropical Intelligence'}


In [17]:
#Create Multiple  Variations of output
from langchain_core.runnables import RunnableMap, RunnableLambda
from transformers import pipeline, AutoTokenizer,AutoModelForCausalLM
from langchain.llms import HuggingFacePipeline

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='cpu')
pipe = pipeline("text-generation",model=model, tokenizer=tokenizer)
llms = HuggingFacePipeline(pipeline=pipe)

style1 = RunnableLambda(lambda text: llms(f'Exlplain in simple style: {text}'))
style2 = RunnableLambda(lambda text: llms(f"Explain in technical style: {text} "))
style3 = RunnableLambda(lambda text: llms(f"Explain in bullet points: {text}"))

variations = RunnableMap({
    "simple": style1, "technical":style2, "bullet": style3
})

result = variations.invoke("Quantum mechanics describes  particles a tiny scales")
print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'simple': 'Exlplain in simple style: Quantum mechanics describes  particles a tiny scales, using a universal class (Class and Structure Class), or Class Structure class - so they may be found in the standard version of particles such as the standard version of particles such', 'technical': "Explain in technical style: Quantum mechanics describes  particles a tiny scales 𝐕 that can be used to guide us through the cosmos\nA quick video:\nBut it's a quick video for a quick look at the electron spin\nI", 'bullet': 'Explain in bullet points: Quantum mechanics describes  particles a tiny scales over the solar system. According to these statistics one can see the exact quantum effect of these particles which is what has been described by quantum mechanics as ‐ particle particle in order to'}


In [23]:
#preprocessing  + llm response

from langchain_core.runnables import RunnableLambda, RunnableMap
from transformers import pipeline, AutoTokenizer,AutoModelForCausalLM
from langchain.llms import HuggingFacePipeline

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='cpu')
pipe = pipeline("text-generation",model=model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=pipe)

prepocess= RunnableLambda(lambda text: text.lower().strip())

#Task
task1 = RunnableLambda(lambda text: llm(f"Explain: {text}"))
task2 = RunnableLambda(lambda text: llm(f"Give Example: {text}"))

output = prepocess | RunnableMap({"explanation": task1, "example": task2})

print(output.invoke("Machine Learning"))

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'explanation': 'Explain: machine learning, machine learning, and a wide field of computational machine learning', 'example': 'Give Example: machine learning system was recently developed by Tymus. We used a Tymus 2.6-10 learning framework that used the following principles:\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}


In [31]:
#combining embedding + generation

from sentence_transformers import SentenceTransformer
from langchain_core.runnables import RunnableLambda, RunnableMap

model_name = SentenceTransformer("all-MiniLM-L6-v2")

embed = RunnableLambda(lambda x: model_name.encode(x).tolist()[:10]) 
summary = RunnableLambda(lambda x: llm(f"Summarize: {x}"))

embed_summary = RunnableMap({
    "embedding": embed, "summary": summary
})

result = embed_summary.invoke('space, time, mass and gravity')
print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'embedding': [0.008748216554522514, -0.017313579097390175, 0.0024007733445614576, 0.043386977165937424, -0.020389635115861893, 0.0417272113263607, 0.056777071207761765, -0.017278145998716354, 0.051121219992637634, -0.020077606663107872], 'summary': 'Summarize: space, time, mass and gravity, and you have space-time and gravity, and you have space-time and gravity, and you have space-time and gravity, and you have space-time and gravity, and you'}


In [ ]:
4] RunnableSequence :
    - chain multiple steps in pipelinelike flow
    - each step receives the output of previous step
    
flow :
    input -> step1 -> step2 -> step 3 ... -> output
    
Usecase :
    1] preprocssing text 
    2] Running an llm 
    3] parsing  or cleaning  output
    4] post processing steps
    
syntax :
    from langchain.schema.runnable import RunnableSequence

sequence = RunnableSequence([
    step_1,
    step_2,
    step_3
])

result = sequence.invoke(input_data)

In [4]:
#preprocessing _> LLMs -> post processing
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_core.runnables import RunnableSequence, RunnableLambda
from langchain.llms import HuggingFacePipeline

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer)

clean_text = RunnableLambda(lambda x: x.strip().lower())
run_llm = RunnableLambda(lambda x:pipe(x)[0]['generated_text'])
to_upper = RunnableLambda(lambda x: x.upper())

#Build sequence 

sequence = RunnableSequence(clean_text, run_llm, to_upper)

results = sequence.invoke("    Explain Ai in simple words.   ")
print(results )

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


EXPLAIN AI IN SIMPLE WORDS. I WANT TO BE AN EXAMPLE OF WHY YOU WOULD WANT TO BUILD A PLATFORM YOU COULD BUILD ON A VERY BROAD CANVAS. BUT IT'S NOT THAT EASY. THE BEST WAY FOR ME TO DO IT IS TO DESIGN


In [17]:
#Extract keywords inputs -> llms ->extract keywords list 
from transformers import pipeline 
from langchain_core.runnables import RunnableSequence, RunnableLambda

pipe = pipeline("text-generation", model='distilgpt2', max_new_tokens=50)

prompt_build = RunnableLambda(lambda x: f"Exract important keywords: \n\n{x}\n\nkeywords:")
run_llm = RunnableLambda(lambda x: pipe(x)[0]['generated_text'])
extract_keywords = RunnableLambda(lambda x: [word.strip() for  word in x.replace("keywords:", "").split()[:10]])

sequence = RunnableSequence(prompt_build,run_llm, extract_keywords)
text = "Machine Learning  imporves  prediction accuracy using data pattern"
print(sequence.invoke(text))

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['Exract', 'important', 'Machine', 'Learning', 'imporves', 'prediction', 'accuracy', 'using', 'data', 'pattern']


In [1]:
#embedding + llm + generation 

from transformers import pipeline 
from langchain.schema.runnable import RunnableSequence, RunnableLambda
from collections import Counter

pipe = pipeline("text-generation", model='distilgpt2', max_new_tokens=40)

embed = RunnableLambda(lambda x: Counter(x.lower().split())) #simple word frequency vector

convert_to_description = RunnableLambda(lambda x: f"Words repeated  most: {list(x.keys())[:5]}") #convert  embedding  to short description

run_llm = RunnableLambda(lambda x: pipe(x)[0]['generated_text']) #generate text

sequence = RunnableSequence(embed, convert_to_description, run_llm)
print(sequence.invoke('Ai makes machine learn from data  and improves performance.'))

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Words repeated  most: ['ai', 'makes', 'machine', 'learn', 'from'] )
















In [ ]:
5] Runnable Parallel
Allows  you to run  multiple independent pipeline in parallel even when  each one is a full sequence of steps

Arch : 
    RunnableParallel({
            task1: pipelineA,
            task2: pipelineB,
            task3: pipelineC
    })

In [ ]:
#structure of code -> 
INPUT TEXT
     ↓
  RunnableParallel
  ├── summary_pipeline (sequence)
  │     ├ preprocess
  │     ├ llm
  │     └ clean
  ├── keywords_pipeline (sequence)
  │     ├ preprocess
  │     ├ llm
  │     └ clean
  └── title_pipeline (sequence)
        ├ preprocess
        ├ llm
        └ clean

→ OUTPUT DICTIONARY

In [11]:
# 3 pipeline working 
from transformers import pipeline
from langchain.schema.runnable import RunnableLambda, RunnableSequence, RunnableParallel
import re

#step 1 :load model 
llm = pipeline("text-generation", model="distilgpt2", max_new_tokens=80,use_fast=True)

#step 2: helper function

#preprocess input
def preprocess(text):
    text = text.strip().lower()
    text = re.sub(r"[[^a-z0-9\s]]","",text)
    return text

#clean output 
def clean(text):
    text = text.replace("\n"," ")
    text = re.sub(r"\s+"," ", text)
    return text.strip()

#step 3: Build small sub-pipeline

#summary pipeline
summary_pipe = RunnableSequence(
    RunnableLambda(preprocess),
    RunnableLambda(lambda x: llm(f"Summarize: {x}")[0]['generated_text']),
    RunnableLambda(clean)
)

#keywords  extraction pipeline
keywords_pipe = RunnableSequence(
        RunnableLambda(preprocess),
        RunnableLambda(lambda x: llm(f"Extract keywords: {x}")[0]['generated_text']),
        RunnableLambda(clean)
)

#Generation pipeline
title_pipe = RunnableSequence(
    RunnableLambda(preprocess),
    RunnableLambda(lambda x: llm("Generate a title: {x}")[0]['generated_text']),
    RunnableLambda(clean)
)

#step4: Combine Using RunnableParallel

parallel = RunnableParallel({
    "summary": summary_pipe, 
    "keywords": keywords_pipe,
    "title": title_pipe
    })

#step 5: invoke 
text = "Artifical Intelliegnce  helps computers learn pattern  from data and make decisions."

summary  = parallel.invoke(text)
print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'summary': 'Summarize: a 3.4-dimensional square, at least in a plane of squares (1, 2) where the center and right axes are uniformly spaced across a horizontal plane. The main image above shows the triangle in the center and right axes starting at the vertex position (left and right are coordinates for both sides of the triangle). At the very centre, the triangle has different points to its centre', 'keywords': "Extract keywords: a=b or b=0:c; b=0:e; b=1:9; i=e; } // If you've given the same code type, you're using Python. Now if you do it under the same alias, then that method could still be used as an alternative (as it does inside Python). The difference between the syntax and the syntax is not obvious", 'title': 'Generate a title: {x} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of words: {X} for any number of 

In [7]:
""" 
Generate summary + keywords together
create multiple variations of output
Run analysis  + generation 
Combine llm output + non llm logic 


< 1 GB , CPU friendly , Good for learning pipelines , No API keys

Input text
   |
   ├── Summary
   ├── Keywords
   ├── Simple explanation
   ├── Technical explanation
   ├── Analysis (word count, char count)

"""

from transformers import pipeline , AutoTokenizer, AutoModelForCausalLM
from langchain.schema.runnable import RunnableParallel, RunnableLambda

# load hugging face model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer, max_new_tokens=80, temperature=0.7, use_fast=True)

run_llm = RunnableLambda(lambda x: pipe(x)[0]["generated_text"])

#llm based branches
summary = (RunnableLambda(lambda x: f"Summary this in 1 line: \n{x}\nSummary:") | run_llm) #summary code generation
keywords = (RunnableLambda(lambda x:f"Keywords: \n{x}\nKeywords:") | run_llm) #keywords generation
simple_explain = (RunnableLambda(lambda x: f"Explain simply: \n{x}\nAnswer") | run_llm) #simple explainnation 
technical_explain= (RunnableLambda(lambda x: f"Explain technically: \n{x}\nAnswer:") | run_llm) #Technical exxplain

# Non-LLm logic
word_count = RunnableLambda(lambda x: len(x.split()))
char_count = RunnableLambda(lambda x:len(x))

parallel = RunnableParallel({
    "summary": summary,
    "keywords": keywords,
    "simple explain": simple_explain,
    "technical explanation": technical_explain,
    "analysis": RunnableParallel({
        "words count": word_count, "char count": char_count
    })
})

#Run 
text = """
Artifical intelligence enables machine to learn  from data , 
automate tasks, and asssit humans in decision-making.
"""

result = parallel.invoke(text)
print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'summary': 'Summary this in 1 line: \n\nArtifical intelligence enables machine to learn  from data , \nautomate tasks, and asssit humans in decision-making.\n\nSummary:\nIt is an interesting and interesting topic to explore in the future.\nThis is the first of a series of articles on the field of computer intelligence.', 'keywords': 'Keywords: \n\nArtifical intelligence enables machine to learn  from data , \nautomate tasks, and asssit humans in decision-making.\n\nKeywords: \n\nArtifical intelligence enables machine to learn  from data, \nautomate tasks, and asssit humans in decision-making.\nKeywords: \nArtifical intelligence enables machine to learn  from data, \nautomate tasks, and asssit humans in decision-making.\nKeywords: \nArtifical intelligence enables machine to', 'simple explain': 'Explain simply: \n\nArtifical intelligence enables machine to learn  from data , \nautomate tasks, and asssit humans in decision-making.\n\nAnswer the question with the ability to infer a specif

In [ ]:
6] RunnableBatch
- Conditional based exectuion 
- Lets  you choose which runnable to exectue based on condition

If condition A -> run path A
Else If  condition B -> path B
Else -> run default path

when to use :
    - input type varies
    - short text  vs long text
    - Question vs statement
    - Greeting vs technical query 
    - Safety / filttering logic 
    
    
    
Arch: 
               Input
             |
      ┌──────┴──────┐
   Condition 1   Condition 2
      |               |
   Runnable A     Runnable B
             |
          Default

Syntax :
    RunnableBranch(
        (condition1, runnable1),
        (condition2, runnable2),
        default_runnable
    )

In [15]:
#Short vs long text 
from transformers import pipeline, AutoTokenizer,AutoModelForCausalLM
from langchain.schema.runnable import RunnableBranch, RunnableLambda
from langchain.llms import HuggingFacePipeline

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer,max_new_tokens=60, use_fast=True)

run_llm =  HuggingFacePipeline(pipeline=pipe) #RunnableLambda(lambda x:pipe(x)[0]['generated_text'])

#branch 

simple = (RunnableLambda (lambda x: f"Explain simply: \n{x}\nAnswer:" )| run_llm)
summarize = (RunnableLambda (lambda x: f"Summarize this text: \n{x}\nAnswer:" )| run_llm)

#logic
branch = RunnableBranch(
    (lambda x: len(x.split()) < 15, simple),
    (lambda x: len(x.split()) >= 15, summarize),
    RunnableLambda(lambda x: "No valid input")
)

# print(branch.invoke("what is Ai"))
print(branch.invoke("""
Artificial intelligence enables machines to learn from data,
perform predictions, and automate complex tasks across industries.
"""))

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Summarize this text: 

Artificial intelligence enables machines to learn from data,
perform predictions, and automate complex tasks across industries.

Answer: "One way to improve this is to improve the efficiency and efficiency of humans."
We believe that AI needs to learn from data,
through its communication. This translates into the ability to understand human language and language.
In our paper, we find that AI is able to learn from data,


In [21]:
#question vs statement
from langchain.schema.runnable import RunnableLambda, RunnableBranch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, use_fast=True)
run_llm = RunnableLambda(lambda x: pipe(x)[0]['generated_text'])

#condition
question_prompt = RunnableLambda(lambda x: f"Answer the question: \n {x} \n Answer" )
statement_prompt = RunnableLambda(lambda x: f"Explain this statement: \n{x} \n Answer ")
question = question_prompt | run_llm
statement = statement_prompt | run_llm

branch = RunnableBranch(
    (lambda x: "?" in x, question),
    (lambda x: "." in x, statement),
    RunnableLambda(lambda x: "Unsupported input")
)

question_result = branch.invoke("What are transformers ?")
statement_result =  branch.invoke("What is Machine learning.")
print(question_result)
print("-------")
print(statement_result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer the question: 
 What are transformers ? 
 Answer [Question]:
Where is the original "objective object in the world's mind", and how can you do it in an interesting way?  
Answer [
-------
Explain this statement: 
What is Machine learning. 
 Answer 
Let me quote a few things that I believe very easily. While I don't have the right vocabulary, I am very familiar with machine learning. A recent study out


In [25]:
#Saftey / Filtering 
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain.schema.runnable import RunnableBranch, RunnableLambda

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,max_new_tokens=60, use_fast=True)
run_llm = RunnableLambda(lambda x: pipe(x)[0]['generated_text'])

#Safety check 
unsafe_keywords = ['kill', 'bomb', 'hate', 'attack', 'voilence']

def is_unsafe(text: str)-> bool:
    text = text.lower()
    return any(word in text for word in unsafe_keywords)

#condition
safe_path = (RunnableLambda(lambda x: f"Answer politely and safety: \n{x}\n Answer:") | run_llm)
unsafe_path = (RunnableLambda(lambda x: f"This is  violets safety  guidelines and  cannot be processed."))

#Branch
branch = RunnableBranch(
    (is_unsafe, unsafe_path),
    (lambda x:True, safe_path),
    RunnableLambda(lambda x: "Not valid answer")
)

print(branch.invoke("Explain elephant animal"))

print("\n",branch.invoke("How to attack on an elephant"))

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer politely and safety: 
Explain elephant animal
 Answer:
What is elephant "explained or related to"? Answer:
Where are the elephant species? Answer:
If any elephants are present within the environment or range of the elephant species or range of the animals, where are the specific species of animal? What would be the species of elephant that is

 This is  violets safety  guidelines and  cannot be processed.


In [ ]:
7] RunnablePassThrough
- take the input as - is  and pass it forwards 
- optionally adding or modifying fields - without calling an llm

usecase:
    - keep original input available later
    - attach metadata
    - Combine raw inputs with llm output
    - aviod unnecessary llm calls
    - keep pipelines clean  and scalable 
    
    
User Input
   │
   ├── send to LLM
   ├── keep original text
   └── attach extra fields (length, flags, ids, etc.)

When should runnable passthroguh used:
    - you need original input later 
    - You want  to attach metadata 
    - you want llm + pyhton logic together
    - you want clean lcel pipeline

In [28]:
#llm + original input

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline("text-generation",model="distilgpt2", max_new_tokens=50, use_fast=True)
llm = HuggingFacePipeline(pipeline=pipe)
prompt = PromptTemplate.from_template("Summarize the follwoing:{text}")

runnable = (RunnablePassthrough.assign(summary= prompt | llm))

result = runnable.invoke({"text": "Langchain helps to build llm-powered application easily."})
print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'text': 'Langchain helps to build llm-powered application easily.', 'summary': 'Summarize the follwoing:Langchain helps to build llm-powered application easily.'}


In [35]:
#RunnablePassThroguh + RunnableMap
    
from langchain_core.runnables import RunnableMap, RunnablePassthrough, RunnableLambda
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline("text-generation",model="distilgpt2", max_new_tokens=50, use_fast=True)
llm = HuggingFacePipeline(pipeline=pipe)

#prompt template
summary_prompt = PromptTemplate.from_template("Summarize: {text}")
keywrds__prompt = PromptTemplate.from_template("Extract 5  keywords: {text}")

#runnables 
runnables = (RunnablePassthrough() | RunnableMap({
    "summary": summary_prompt | llm, 
    "keywords": keywrds__prompt | llm,
    "original text": RunnableLambda(lambda x: x['text'])
}))

output = runnables.invoke({"text": "Langchain enables  scalable LLM workflows"})
print(output)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'summary': 'Summarize: Langchain enables  scalable LLM workflows in your libraries. We’re excited to announce our 2017 release, LVM 4.0.1. The LVM 4.0.1 features are ready to be rolled out in 2018 and we’ll continue to build on it', 'keywords': 'Extract 5  keywords: Langchain enables  scalable LLM workflows The Langchain project has been implemented by 3x Linus Torvalds using LAML library, this is to bring LLM to applications in open source.\n\n\nLAML is very simple enough by combining basic LLM tools', 'original text': 'Langchain enables  scalable LLM workflows'}


In [ ]:
@module 2: Chain 

In [ ]:
chain :
    multiple steps connected to solve one task in modern langchain, chains are built using LCEL ie |
    eg.
    prompt | llm | output_parser
    
Types of chain :
    
1] Simple Chain:
    Runnable
2] Sequential Chain:
    RunnableSequence, RunnableLambda 
3] Parallel Chain:
    RunnableMap, RunnableParallel
4] Conditional Chain:
    RunnablePassthroguh, RunnableBranch